# Шаг 3. Сглаживание: какой фильтр лучше убирает шум?

Камеры на производстве дают шумные кадры: пыль, блики, битые пиксели.
Перед распознаванием шум убирают **сглаживающими фильтрами**. Все они работают одинаково по идее:
для каждого пикселя берут окно соседей (например, 5 × 5) и заменяют пиксель на «итог» этого окна.
Отличаются они тем, **как** считается этот итог.

Мы искусственно испортим фото шумом **«соль и перец»** (случайные белые и чёрные точки)
и сравним четыре фильтра. А ещё повторим `cv2.blur` вручную через **свёртку** — основу свёрточных нейросетей (сессия 5).

**Структура ноутбука (алгоритм):**
1. Импортируем OpenCV, NumPy и помощники.
2. Загружаем и уменьшаем фото.
3. Добавляем шум: 3% пикселей делаем чёрными, 3% — белыми.
4. Применяем четыре фильтра: среднее, Гаусс, медиана, билатеральный.
5. Повторяем среднее вручную: своё ядро свёртки + `cv2.filter2D`, сверяем с `cv2.blur`.
6. Собираем коллаж, сохраняем в `smoothing.jpg` и сравниваем.

In [ ]:
# Настройка для Google Colab. На своём компьютере эта ячейка ничего не делает.
# В Colab нужна папка machine_vision на Google Диске (в ней session01–session05 и dataset).
import os
import sys

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive:
    drive.mount("/content/drive")
    COLAB_ROOT = "/content/drive/MyDrive/machine_vision"
    os.chdir(f"{COLAB_ROOT}/session02")  # здесь лежит common.py и сохраняются результаты
    sys.path.insert(0, os.getcwd())

## 1. Импорт библиотек и общих помощников

- `cv2` — библиотека **OpenCV**: чтение, обработка и показ изображений.
- `numpy` (`np`) — работа с массивами чисел. Картинка в OpenCV — это и есть массив NumPy.
- `common` — наш модуль из этой же папки (`common.py`) с маленькими помощниками:

| Имя | Что делает |
|-----|------------|
| `PIPE` | путь к фотографии трубы из датасета (`dataset/images/train/...`) |
| `load(path)` | прочитать картинку (работает и с русскими буквами в пути) |
| `save(path, img)` | сохранить картинку в файл |
| `show_nb(img)` | показать картинку прямо в ноутбуке, под ячейкой |
| `label(img, text)` | подписать картинку чёрной полосой сверху (только латиница) |
| `grid(images, cols)` | склеить несколько картинок одинакового размера в таблицу |
| `fit(img, w, h)` | уменьшить картинку, если она не влезает в заданный размер |

> Ноутбук нужно запускать из папки `session02`, иначе `import common` не найдёт модуль.

In [ ]:
import cv2
import numpy as np

from common import PIPE, grid, label, load, save, show_nb

## 2. Загружаем картинку

In [ ]:
img = cv2.resize(load(PIPE), (640, 360))

## 3. Добавляем шум «соль и перец»

Алгоритм:
1. `rng = np.random.default_rng(seed=0)` — генератор случайных чисел. Фиксированный `seed` даёт **одинаковый** шум при каждом запуске, поэтому результаты можно сравнивать.
2. `rng.random(img.shape[:2])` — для каждого пикселя случайное число от 0 до 1 (массив размером высота × ширина).
3. Где число `< 0.03` — пиксель делаем чёрным (перец), где `> 0.97` — белым (соль).

`noisy[chance < 0.03] = 0` — это **булева маска**: выражение `chance < 0.03` даёт массив `True/False`,
и присваивание срабатывает только там, где `True`. Сразу во всех трёх каналах.

In [ ]:
# Шум: 3% пикселей чёрные, 3% белые
rng = np.random.default_rng(seed=0)
noisy = img.copy()
chance = rng.random(img.shape[:2])
noisy[chance < 0.03] = 0
noisy[chance > 0.97] = 255
show_nb(noisy)

## 4. Четыре фильтра

Размер окна `k = 5` — всегда **нечётный**, чтобы у окна был центральный пиксель.

| Фильтр | Функция | Итог окна |
|--------|---------|-----------|
| Среднее | `cv2.blur` | среднее арифметическое всех 25 пикселей |
| Гаусс | `cv2.GaussianBlur` | взвешенное среднее: центр важнее краёв (веса — «колокол» Гаусса); `0` — подобрать ширину колокола по размеру окна |
| Медиана | `cv2.medianBlur` | значения окна сортируются, берётся среднее по порядку (13-е из 25) |
| Билатеральный | `cv2.bilateralFilter(img, 9, 75, 75)` | усредняет только **похожих по цвету** соседей, поэтому границы не размываются; 9 — диаметр окна, 75 и 75 — допуск по цвету и по расстоянию |

Почему медиана хороша против «соли и перца»: одиночная белая точка (255) при сортировке окна
улетает в самый конец и в медиану не попадает. А в среднее она вносит вклад `255 / 25 ≈ 10` — точка просто становится серым пятном.

In [ ]:
k = 5  # размер окна фильтра: 5×5 пикселей, всегда нечётный

mean = cv2.blur(noisy, (k, k))                    # среднее по окну
gauss = cv2.GaussianBlur(noisy, (k, k), 0)        # среднее, но центр важнее краёв
median = cv2.medianBlur(noisy, k)                 # медиана: середина отсортированного окна
bilateral = cv2.bilateralFilter(noisy, 9, 75, 75)  # сглаживает, но бережёт границы

## 5. Что внутри `cv2.blur`: свёртка своими руками

**Свёртка** — это операция, при которой маленькая матрица весов (**ядро**) скользит по картинке,
в каждой позиции её веса умножаются на пиксели под ней, а произведения складываются.

Для среднего ядро 5 × 5 состоит из одинаковых весов `1/25`:
сумма `1/25 · p₁ + 1/25 · p₂ + … + 1/25 · p₂₅` и есть среднее.

`cv2.filter2D(img, -1, kernel)` применяет любое ядро (`-1` — тип результата как у исходника).
Сравниваем со встроенным `cv2.blur`: максимальная разница должна быть 0 (или 1 из-за округления).
Перед вычитанием переводим в `int`, чтобы `uint8` не «завернулся» при отрицательной разнице.

> В свёрточных нейросетях (сессия 5) веса ядер не задаются вручную, а **подбираются обучением**.

In [ ]:
# Что внутри cv2.blur: окно 5×5, каждый вес 1/25 — это и есть свёртка (сессия 5)
kernel = np.ones((k, k)) / (k * k)
manual = cv2.filter2D(noisy, -1, kernel)
print("Ядро:\n", kernel)
print("Разница между cv2.blur и своей свёрткой:", int(np.abs(manual.astype(int) - mean).max()))

## 6. Сравниваем все фильтры

In [ ]:
collage = grid([
    label(noisy, "noise"), label(mean, "blur (mean)"), label(manual, "filter2D: own kernel"),
    label(gauss, "GaussianBlur"), label(median, "medianBlur"), label(bilateral, "bilateralFilter"),
])

save("smoothing.jpg", collage)
show_nb(collage, 1920, 1080)
print("Вопрос: почему медиана справилась с точками лучше среднего?")

## Попробуйте сами

1. Увеличьте `k` до 9 и 15. Что происходит с шумом и с чёткостью маркировки?
2. Увеличьте долю шума с 3% до 10%. Справляется ли медиана с окном 5?
3. Замените шум на гауссов: `noisy = np.clip(img + rng.normal(0, 25, img.shape), 0, 255).astype(np.uint8)`. Какой фильтр теперь лучше?
4. Составьте своё ядро «повышения резкости» `[[0,-1,0],[-1,5,-1],[0,-1,0]]` и примените его через `filter2D` к `img`.